# P3-F6 — 6 run con lai: huan luyen + eval da nhanh trong MOT lan chay

Doi `JOB` o Cell 2 roi **Run All** (~3h/job). Ra ca ket qua huan luyen lan bang
img/txt/fuse, khong phai tai/upload checkpoint giua chung.

| JOB | Bo | Muc quen | Ablation | Config | ~gio |
|---|---|---|---|---|---:|
| `p3_m6`    | MIMIC | 6 %  | – | advanced + F6 | 2,7 |
| `p3_m10`   | MIMIC | 10 % | – | advanced + F6 | 2,7 |
| `p3_iu`    | IU    | 3 %  | – | loku_iu + F6 + lr 2e-4/clip 0 | 3,0 |
| `abl_fila` | MIMIC | 3 %  | w/o Fisher+FILA | advanced + F6 | 2,6 |
| `abl_ihl`  | MIMIC | 3 %  | w/o IHL | advanced + F6 | 2,7 |
| `abl_mumr` | MIMIC | 3 %  | w/o MU/MR | advanced + F6 | 2,7 |

## Phan cong 4 account

**Vong 1**: acc1 = `p3_m6` · acc2 = `p3_m10` · acc3 = `p3_iu` · acc4 = `abl_fila`
**Vong 2**: acc1 = `abl_ihl` · acc2 = `abl_mumr`

Tong ~16,4h GPU nhung cho thuc te ~6h.

## Cau hinh F6 (da chot tren MIMIC 3 %)

    lambda_IHL 5.0 · lambda_CE 0.25 · lambda_KD 0
    w_UR = w_UU = 1/3 · w_MU = w_MR = 1/6        (scheme uni_nokd)
    loku_subtract_scale 1.0 · loku_image_subtract_scale 1.0
    lora_image_last_k_blocks 3 · lora_image_include_fc1 0
    lora_extra = attention.output.dense|intermediate.dense|output.dense
    lr 2e-4 · grad_clip 0 · r=8 · alpha=16 · 30 epoch · bs 16

Ket qua MIMIC 3 %: Df-AUC 0.521 (gold 0.498) · Dt-AUC 0.670 · MIA 0.269 ·
forget/test-CE 4.515/2.360 · 1 488 896 tham so (1,30 %) · T_core 0,54 h.

## Ba cho de hong am tham — deu da chan trong code

1. **`abl_ihl` khong duoc override `lambda_ihl`.** `forgetmi_p3_cand` dat `lambda_ihl=0`
   trong `extra` (dong 108) nhung `apply_overrides` chay SAU (116-117) -> de nguyen la
   ablation bien thanh chinh F6.
2. **`abl_fila` phai bao lai `loku_random_init=1` luc EVAL.** No duoc huan luyen khong co
   FILA; dung lai ma quen co nay thi nen W* khac luc train, moi so sai ma khong bao loi.
3. **Hai he so tru FILA phai khai lai luc eval.** Chung khong suy duoc tu ten khoa
   checkpoint (khac `lora_extra` / `lora_image_last_k_blocks`) ma lai quyet dinh
   `W* = W - gamma*B*A*`.

## Luu y ve nhanh van ban

`theta_og` va `theta_re` do bai bao phat hanh **khong cung quy trinh huan luyen**: than
BERT lech tuong doi ~1.0, `text_model.classifier` lech 7,8 lan (og 0.765 ~ muc khoi tao,
re 5.97). Vi moi phuong phap go bo deu khoi tu `theta_og`, nhanh van ban cua chung se
quanh muc ngau nhien — do la ke thua, khong phai loi phuong phap. Khi lap bang da nhanh:
**moc so sanh la `theta_og`, KHONG phai `theta_re`.**


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
for f in ['training/forgetmi_p3_cand.py','training/eval_multimodal.py','training/adv_common.py']:
    assert os.path.exists(f), f'Thieu {f} -> git push code moi truoc'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))

print('\n--- tu kiem eval_multimodal (30 giay, truoc khi ton 2,7h GPU) ---')
subprocess.run(['python','tools/test_eval_multimodal.py'],check=True)


In [ ]:
# Cell 2: CHON JOB + path
import glob, os

JOB    = 'p3_m6'    # p3_m6 | p3_m10 | p3_iu | abl_fila | abl_ihl | abl_mumr
SEED   = 42
EPOCHS = 30

# job -> (dataset, forget %, ablation)
JOBS = {'p3_m6'   : ('mimic', 6,  'none'),
        'p3_m10'  : ('mimic', 10, 'none'),
        'p3_iu'   : ('iu',    3,  'none'),
        'abl_fila': ('mimic', 3,  'fisher_fila'),
        'abl_ihl' : ('mimic', 3,  'ihl'),
        'abl_mumr': ('mimic', 3,  'mu_mr')}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
DATASET, PCT, ABLATE = JOBS[JOB]

# ===== F6 — cau hinh da chot tren MIMIC 3% =====
F6 = {'lambda_ihl': 5.0, 'lambda_ce': 0.25,
      'loku_subtract_scale': 1.0, 'loku_image_subtract_scale': 1.0,
      'lora_image_last_k_blocks': 3, 'lora_image_include_fc1': 0}

OVR_METHOD = dict(F6)
if DATASET == 'iu':
    # config_loku_iu dung lr 5e-4 + clip 1.0; khoi phuc dung cau hinh khoa tu MIMIC.
    OVR_METHOD.update({'learning_rate': 2.0e-4, 'grad_clip': 0.0})
if ABLATE == 'ihl':
    # BAT BUOC: p3_cand dat lambda_ihl=0 trong `extra` (dong 108) nhung apply_overrides
    # chay SAU (116-117) -> de lambda_ihl=5.0 o day la ablation bi ghi de am tham.
    OVR_METHOD.pop('lambda_ihl')

RID = f'{JOB}_s{SEED}'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

if DATASET=='mimic':
    CONFIG='config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add Input: forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{PCT}per.csv'
else:
    CONFIG='config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add Input: forget-mi-data-iu + forget-mi-models-iu(+ -re) + raddar'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0])
    IMG=(glob.glob(os.path.join(RAD,'**','images_normalized'),recursive=True) or [RAD])[0]
    SPLIT=(glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True)+
           glob.glob('/kaggle/input/**/iu-split.csv',recursive=True))[0]
    FORGET=(glob.glob(os.path.join(DATA,'**',f'forget_set_{PCT}per_iu.csv'),recursive=True)+
            glob.glob(f'/kaggle/input/**/forget_set_{PCT}per_iu.csv',recursive=True))[0]

for n,p in {'BASE':BASE,'GOLD':GOLD,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

OUT     = f'/kaggle/working/kltn_{JOB}_s{SEED}'
OD      = f'{OUT}/{RID}'
CKPT    = f'{OD}/checkpoints/latest.pt'
R_TRAIN = f'/kaggle/working/results_{JOB}.csv'
R_MM    = f'/kaggle/working/results_multimodal_{JOB}.csv'
HIST    = f'/kaggle/working/perepoch_{RID}.csv'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'use_noise':1}
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT}

# Khoa phai khai lai luc dung lai W* tu checkpoint chi-chua-LoRA.
REBUILD={k:F6[k] for k in ['loku_subtract_scale','loku_image_subtract_scale',
                           'lora_image_last_k_blocks','lora_image_include_fc1']}
if ABLATE=='fisher_fila':
    # Duoc huan luyen KHONG co FILA -> dung lai ma quen co nay thi nen W* khac luc train.
    REBUILD['loku_random_init']=1

print('JOB      :',JOB,'|',DATASET,PCT,'% | ablate:',ABLATE)
print('config   :',CONFIG)
print('run id   :',RID)
print('ckpt se o:',CKPT)
print('\n--- overrides ---')
for k,v in OVR_METHOD.items(): print(f'   {k:32} = {v}')
if ABLATE=='ihl':
    print('   (lambda_ihl CO Y KHONG override -> ablation tu dat = 0)')
print('\n--- rebuild khi eval ---')
for k,v in REBUILD.items(): print(f'   {k:32} = {v}')


In [ ]:
# Cell 3: HUAN LUYEN (30 epoch)
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

ovr=dict(COMMON); ovr.update(MORE)
ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,'results_csv_path':R_TRAIN,
            'ce_selector':1,'s4_delta':0.15,'history_csv_path':HIST})
ovr.update(OVR_METHOD)
cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
     '--scheme','uni_nokd','--ablate',ABLATE,'--fresh','--override',
     ','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\nTRAIN {RID}\n'+'='*72)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True)
    print(f'OK train  {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL train rc=',e.returncode)
print('checkpoint ton tai:',os.path.exists(CKPT))


In [ ]:
# Cell 4: EVAL DA NHANH (img / txt / fuse) tren checkpoint vua tao
# FP16 truoc (cung do chinh xac voi og/re da do). Neu txt/fuse ra NaN -> tu chay lai FP32.
import os, subprocess, time
import pandas as pd

def run_eval(label, extra):
    ovr=dict(COMMON); ovr.update(MORE); ovr.update(REBUILD)
    ovr['output_dir']=f'{OUT}/_mm'; ovr.update(extra)
    cmd=['python','training/eval_multimodal.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','p3_lora','--model_path',CKPT,
         '--out_csv',R_MM,'--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('='*72+f'\nEVAL {label}\n'+'='*72)
    t0=time.time()
    try:
        subprocess.run(cmd,env=env,check=True)
        print(f'OK {label}  {(time.time()-t0)/60:.1f} phut'); return True
    except subprocess.CalledProcessError as e:
        print(f'FAIL {label} rc={e.returncode}'); return False

if not os.path.exists(CKPT):
    print('Khong co checkpoint -> bo qua eval. Xem lai Cell 3.')
else:
    run_eval(JOB, {})
    need_fp32=False
    if os.path.exists(R_MM):
        d=pd.read_csv(R_MM); d=d[d['label']==JOB]
        need_fp32=bool(d[d['view'].isin(['txt','fuse'])]['Df_AUC'].isna().any())
    if need_fp32:
        print('\ntxt/fuse ra NaN o FP16 -> chay lai FP32...')
        run_eval(f'{JOB}_fp32', {'eval_autocast':0})
    else:
        print('\nFP16 du dung, khong can FP32.')


In [ ]:
# Cell 5: BANG KET QUA + TU KIEM
import os
import pandas as pd
pd.set_option('display.width',250)

print(f'===== HUAN LUYEN {JOB} (E30 = checkpoint_kind "last") =====')
train_row=None
if os.path.exists(R_TRAIN):
    dt=pd.read_csv(R_TRAIN)
    cols=[c for c in ['id','checkpoint_kind','selected_epoch','Forget_AUC','Forget_Macro_F1',
                      'Test_AUC','Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce',
                      '1_minus_Sim','trainable_params','trainable_ratio','core_seconds',
                      'lambda_ihl','w_mu','w_mr'] if c in dt.columns]
    print(dt[cols].to_string(index=False))
    last=dt[dt['checkpoint_kind']=='last']
    if len(last): train_row=last.iloc[-1]
else:
    print('chua co',R_TRAIN)

print('\n===== EVAL DA NHANH =====')
if os.path.exists(R_MM):
    dm=pd.read_csv(R_MM)
    c=['label','view','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper',
       'member_ce','nonmember_ce','forget_ce']
    print(dm[[x for x in c if x in dm.columns]].to_string(index=False))
else:
    print('chua co',R_MM)

print('\n===== TU KIEM =====')
if ABLATE=='ihl' and train_row is not None and 'lambda_ihl' in train_row:
    v=float(train_row['lambda_ihl'])
    print(f'  lambda_ihl = {v}  ' + ('DUNG (ablation da tat IHL)' if v==0
          else '*** SAI: ablation bi ghi de, KHONG dung ket qua nay ***'))
if ABLATE=='mu_mr' and train_row is not None and 'w_mu' in train_row:
    print(f"  w_mu={train_row['w_mu']}  w_mr={train_row['w_mr']}  "
          + ('DUNG (da tat MU/MR)' if float(train_row['w_mu'])==0 else '*** SAI ***'))
if train_row is not None and os.path.exists(R_MM):
    img=pd.read_csv(R_MM)
    img=img[(img['label']==JOB) & (img['view']=='img')]
    if len(img):
        r=img.iloc[-1]
        for name,a,b in [('Df-AUC', r['Df_AUC'], train_row['Forget_AUC']),
                         ('Dt-AUC', r['Dt_AUC'], train_row['Test_AUC']),
                         ('MIA',    r['MIA'],    train_row['MIA']),
                         ('forget-CE', r['forget_ce'], train_row['forget_ce'])]:
            ok = abs(float(a)-float(b)) < 0.005
            print(f'  {name:10} eval {float(a):7.4f}  vs  train {float(b):7.4f}   '
                  + ('TRUNG' if ok else '*** LECH -> dung tin so txt/fuse ***'))

print('\n===== MOC DA CO (MIMIC 3%, nhanh anh, E30) =====')
print('  theta_og   Df 0.731  Dt 0.695  MIA 0.657  fce 1.966  tce 2.108')
print('  gold       Df 0.498  Dt 0.615  MIA 0.423  fce 4.736  tce 3.046')
print('  Forget-MI  Df 0.623  Dt 0.648  MIA 0.627  fce 3.578  tce 2.906')
print('  P3-F6      Df 0.521  Dt 0.670  MIA 0.269  fce 4.515  tce 2.360')
print('  (6%/10%/IU co moc rieng — xem KET_QUA_THUC_NGHIEM.md)')

print(f'\nTAI VE: results_{JOB}.csv - results_multimodal_{JOB}.csv - perepoch_{RID}.csv')
print(f'        + kltn_{JOB}_s{SEED}/**/checkpoints/latest.pt')
print('NHO Save Version ngay khi chay xong.')
